# Resolve Schema

**input**
```
+schema_path: str
```

**methods**
```
+read_json(schema_path: str): dict
+split_json(schema: dict): list
+resolver(entity1: dict, entity2: dict): dict
+resolve_defs(terms: dict, defs: dict) : dict
+ node_order(schema: dict): list
+resolve_nodes(nodeList: list, splitJsonList: list): list
+recombine_nodes(resolvedList: list) : dict
```

In [ ]:
import gen3_validator
from gen3_validator.logging_config import setup_logging
setup_logging()

In [ ]:
# pulling manifest
!aws s3 cp s3://ausdiab-data-receive-bucket/data/2025-01-31_AusDiab_ACDC_Data_Transfer/manifest.xlsx ../data/restricted/ausdiab_lipid_manifest.xlsx

In [ ]:
# pulling schema
!aws s3 cp s3://gen3schema-cad-staging-biocommons.org.au/cad.json ../schema/gen3_schema.json

## Reading in xlsx data and writing to json
- xlsx data comes from xlsx manifest file created from acdc_submission_template

In [ ]:
# ResolverClass = gen3_validator.ResolveSchema(schema_path = "../schema/gen3_test_schema.json")
xlsxData = gen3_validator.ParseXlsxMetadata(xlsx_path = "/Users/harrijh/projects/gen3-data-validator/data/lipid_metadata_example.xlsx", skip_rows=1)
xlsxData.parse_metadata_template()
xlsxData.write_dict_to_json(xlsx_data_dict=xlsxData.xlsx_data_dict, output_dir="/Users/harrijh/projects/gen3-data-validator/data/restricted/lipid_metadata_example")

## Testing Linkage

In [ ]:
Data = gen3_validator.ParseData(data_folder_path = "/Users/harrijh/projects/gen3-data-validator/data/restricted/lipid_metadata_example")

In [ ]:
Data.file_path_list

In [ ]:
Data = gen3_validator.ParseData(data_folder_path = "/Users/harrijh/projects/gen3-data-validator/data/restricted/lipid_metadata_example")
Resolver = gen3_validator.ResolveSchema(schema_path = "../schema/gen3_schema.json")
Resolver.resolve_schema()
Linkage = gen3_validator.Linkage(schema_resolver = Resolver, data_parser = Data)

In [ ]:
Resolver = gen3_validator.ResolveSchema(schema_path = "../schema/gen3_schema.json")

In [ ]:
Resolver.nodes

You can also bypass the data attribute in the linkage class and input your own data_map and config_map

In [ ]:
# # Using the Linkage class which has the resolved schema with custom data and config
# config_map = {
#     "samples": {"primary_key": "sample_id", "foreign_key": "subject_id"},
#     "files": {"primary_key": "file_id", "foreign_key": "sample_id"},
#     "subjects": {"primary_key": "subject_id", "foreign_key": "project_id"},
#     "project": {"primary_key": "project_id", "foreign_key": None}
# }

# data_map = {
#     "samples": [
#         {"sample_id": "sample_1", "subject_id": "subject_9"},
#         {"sample_id": "sample_2", "subject_id": "subject_3"},  # Invalid FK
#         {"sample_id": "sample_3", "subject_id": "subject_4"}, # Invalid FK
#         {"sample_id": "sample_4", "subject_id": "subject_5"} # Invalid FK
#     ],
#     "files": [
#         {"file_id": "file_1", "sample_id": "sample_1"},
#         {"file_id": "file_2", "sample_id": "sample_27"}  # Invalid FK
#     ],
#     "subjects": [
#         {"subject_id": "subject_1", "project_id": "project_1"},  
#         {"subject_id": "subject_2", "project_id": "project_2"}, # Missing project 2
#     ],
#     "project": [
#         {"project_id": "project_1"}
#     ]
# }

# Linkage.validate_links(data_map, config_map)

# validation prototype


## Creating the validation class

In [ ]:
import gen3_validator

resolver = gen3_validator.ResolveSchema(schema_path = "../schema/gen3_schema.json")
resolver.resolve_schema()
data = gen3_validator.ParseData(data_folder_path = "../data/restricted/lipid_metadata_example")
validator = gen3_validator.Validate(data_map=data.data_dict, resolved_schema=resolver.schema_resolved)

In [ ]:
validator.validate_schema()

In [ ]:
validator.make_keymap()

In [ ]:
data.data_dict

### Getting nested validation results
- returns a nested dictionary by entity/data node then by the row/index number, and then the validation objects

In [ ]:
validation_dict = validator.validation_result
validation_dict

In [ ]:
validator.list_entities()

In [ ]:
validator.list_index_by_entity("lipidomics_assay")

You can pull out a validation results for a specific entity with

In [ ]:
validator.pull_entity("lipidomics_assay")

You can pull validation results for a specific entity and then a specific index / row

In [ ]:
validator.pull_index_of_entity("lipidomics_assay", "index_1")

# Getting validation stats

In [ ]:
validate_stats = gen3_validator.ValidateStats(validator)
stats_df = validate_stats.summary_stats()
stats_df

# Creating validation summary data

In [ ]:
Summary = gen3_validator.ValidateSummary(validator) 
flattened_results_dict = Summary.flatten_validation_results()
flattened_results_dict

### Converting flattened dict to pandas

In [ ]:
flatten_summary_pd = Summary.flattened_results_to_pd()
flatten_summary_pd

### Collapsing flattened dict to pandas
- This collapsed data frame summarises common validation errors

In [ ]:
collapse_df = Summary.collapse_flatten_results_to_pd()
collapse_df

# Writing validation results to folder

In [ ]:
import os
output_dir = "../data/restricted/ausdiab_lipid_metadata/validation/"
os.makedirs(output_dir, exist_ok=True)


def write_dict_to_json(input_dict, output_dir, filename:str):
    with open(f"{output_dir}/{filename}.json", "w") as f:
        json.dump(input_dict, f)
    print(f"JSON files written to {output_dir}")

write_dict_to_json(validation_dict, output_dir, "validation_dict")
write_dict_to_json(flattened_results_dict, output_dir, "flattened_results_dict")

# Writing pandas df
stats_df.to_csv(f"{output_dir}/stats_df.csv")
flatten_summary_pd.to_csv(f"{output_dir}/flatten_summary_pd.csv")
collapse_df.to_csv(f"{output_dir}/collapse_df.csv")


In [ ]:
# Use this for writing tests

sample_validation_results = {
    'sample': [
        [
            {
                'index': 0,
                'validation_result': 'FAIL',
                'invalid_key': 'freeze_thaw_cycles',
                'schema_path': 'properties.freeze_thaw_cycles.type',
                'validator': 'type',
                'validator_value': 'integer',
                'validation_error': "'10' is not of type 'integer'"
            },
            {
                'index': 0,
                'validation_result': 'FAIL',
                'invalid_key': 'sample_provider',
                'schema_path': 'properties.sample_provider.enum',
                'validator': 'enum',
                'validator_value': ['Baker', 'USYD', 'UMELB', 'UQ'],
                'validation_error': "45 is not one of ['Baker', 'USYD', 'UMELB', 'UQ']"
            },
            {
                'index': 0,
                'validation_result': 'FAIL',
                'invalid_key': 'sample_storage_method',
                'schema_path': 'properties.sample_storage_method.enum',
                'validator': 'enum',
                'validator_value': [
                    'not stored',
                    'ambient temperature',
                    'cut slide',
                    'fresh',
                    'frozen, -70C freezer',
                    'frozen, -150C freezer',
                    'frozen, liquid nitrogen',
                    'frozen, vapor phase',
                    'paraffin block',
                    'RNAlater, frozen',
                    'TRIzol, frozen'
                ],
                'validation_error': "'Autoclave' is not one of ['not stored', 'ambient temperature', 'cut slide', 'fresh', 'frozen, -70C freezer', 'frozen, -150C freezer', 'frozen, liquid nitrogen', 'frozen, vapor phase', 'paraffin block', 'RNAlater, frozen', 'TRIzol, frozen']"
            }
        ],
        [
            {
                'index': 1,
                'validation_result': 'FAIL',
                'invalid_key': 'freeze_thaw_cycles',
                'schema_path': 'properties.freeze_thaw_cycles.type',
                'validator': 'type',
                'validator_value': 'integer',
                'validation_error': "'76' is not of type 'integer'"
            },
            {
                'index': 1,
                'validation_result': 'FAIL',
                'invalid_key': 'sample_storage_method',
                'schema_path': 'properties.sample_storage_method.enum',
                'validator': 'enum',
                'validator_value': [
                    'not stored',
                    'ambient temperature',
                    'cut slide',
                    'fresh',
                    'frozen, -70C freezer',
                    'frozen, -150C freezer',
                    'frozen, liquid nitrogen',
                    'frozen, vapor phase',
                    'paraffin block',
                    'RNAlater, frozen',
                    'TRIzol, frozen'
                ],
                'validation_error': "'In the Pantry' is not one of ['not stored', 'ambient temperature', 'cut slide', 'fresh', 'frozen, -70C freezer', 'frozen, -150C freezer', 'frozen, liquid nitrogen', 'frozen, vapor phase', 'paraffin block', 'RNAlater, frozen', 'TRIzol, frozen']"
            }
        ],
        [
            {
                'index': 2,
                'validation_result': 'PASS',
                'invalid_key': None,
                'schema_path': None,
                'validator': None,
                'validator_value': None,
                'validation_error': None
            }
        ],
        [
            {
                'index': 3,
                'validation_result': 'PASS',
                'invalid_key': None,
                'schema_path': None,
                'validator': None,
                'validator_value': None,
                'validation_error': None
            }
        ]
    ]
}




# Trying to figure out minimum node order

In [ ]:
from gen3_validator.dict import DataDictionary

In [ ]:
dd = DataDictionary(schema_path = "../tests/schema/gen3_test_schema.json")
dd.parse_schema()

In [ ]:
dd.get_nodes()

In [ ]:
dd.generate_node_lookup()

In [ ]:
dd.get_all_node_pairs()

In [ ]:
from collections import defaultdict

def find_all_paths_and_steps(self, edges: list) -> dict:
    """
    Finds all possible paths in a directed graph and calculates the number of steps for each.

    This method builds a graph from the list of edges and performs a Depth-First Search (DFS)
    from each source node (a node with no incoming edges). It records every intermediate
    path and its corresponding length.

    For example, for a graph A -> B -> C, this function will identify and record:
    - ('A', 'B'): 1
    - ('A', 'B', 'C'): 2

    :param edges: A list of tuples, where each tuple is a node pair (upstream, downstream).
    :type edges: list

    :return: A dictionary where keys are tuples representing the paths and values are the
             number of steps (edges) in that path.
    :rtype: dict
    """
    # Step 1: Build the graph and identify source nodes, similar to your method.
    graph = defaultdict(list)
    in_degree = defaultdict(int)
    all_nodes = set()

    for upstream, downstream in edges:
        graph[upstream].append(downstream)
        in_degree[downstream] += 1
        all_nodes.add(upstream)
        all_nodes.add(downstream)
    
    # Identify source nodes (nodes with an in-degree of 0).
    source_nodes = [node for node in all_nodes if in_degree[node] == 0]

    # This dictionary will store the final results.
    all_paths = {}

    # Step 2: Define the recursive DFS traversal function.
    def dfs(current_path):
        """Recursively explores the graph and records paths."""
        node = current_path[-1]

        # Record every path segment that has at least one step.
        if len(current_path) > 1:
            all_paths[tuple(current_path)] = len(current_path) - 1
        
        # If we reach a node with no outgoing edges, the path ends here.
        if node not in graph:
            return
        
        # Continue the traversal for all neighbors.
        for neighbor in graph[node]:
            # To prevent infinite loops in graphs with cycles, we don't revisit nodes
            # already in the current traversal path.
            if neighbor not in current_path:
                dfs(current_path + [neighbor])

    # Step 3: Start the DFS from each identified source node.
    for start_node in source_nodes:
        dfs([start_node])
        
    return all_paths



In [ ]:
find_all_paths_and_steps(dd, edges=dd.get_all_node_pairs())

In [ ]:
{
    'lipidomics_file': [
        {
            "path": [
                "sample",
                "lipidomics_assay",
                "lipidomics_file"
            ],
            "steps": 3
        },
        {
            "path": [
                "sample",
                "lipidomics_file"
            ],
            "steps": 2
        }
    ],
    'metabolomics_file': [
        {
            "path": [
                "sample",
                "metabolomics_assay",
                "metabolomics_file"
            ],
            "steps": 3
        },
        {
            "path": [
                "sample",
                "metabolomics_file"
            ],
            "steps": 2
        }
    ]
}

trying data classes

In [ ]:
# graph_paths.py

from dataclasses import dataclass
from typing import List, Dict, Optional, Any, Set, Tuple
from collections import defaultdict

@dataclass
class PathInfo:
    """
    Data structure representing a single path in a directed graph.

    Attributes
    ----------
    path : List[str]
        The sequence of node names (as strings) representing the path from the root to the destination node.
    steps : int
        The number of steps (edges) in the path. This is typically `len(path) - 1`.
    """

    path: List[str]
    steps: int

def build_graph(
    edges: List[Tuple[str, str]],
    ignore_nodes: Optional[List[str]] = None
) -> Tuple[Dict[str, List[str]], Set[str], Set[str]]:
    """
    Build an adjacency list representation of a directed graph from a list of edges.

    Parameters
    ----------
    edges : List[Tuple[str, str]]
        A list of (upstream, downstream) node pairs representing directed edges in the graph.
    ignore_nodes : Optional[List[str]], optional
        A list of node names to ignore when building the graph. Edges involving these nodes are skipped.
        Defaults to None.

    Returns
    -------
    graph : Dict[str, List[str]]
        The adjacency list representation of the graph, mapping each node to a list of its downstream neighbors.
    all_nodes : Set[str]
        The set of all node names present in the graph (including both upstream and downstream nodes).
    downstream_nodes : Set[str]
        The set of all nodes that appear as downstream nodes in any edge.

    Examples
    --------
    >>> edges = [('A', 'B'), ('B', 'C')]
    >>> build_graph(edges)
    ({'A': ['B'], 'B': ['C']}, {'A', 'B', 'C'}, {'B', 'C'})
    """
    if ignore_nodes is None:
        ignore_nodes = []
    graph = defaultdict(list)
    all_nodes = set()
    downstream_nodes = set()
    for upstream, downstream in edges:
        if upstream in ignore_nodes or downstream in ignore_nodes:
            continue
        graph[upstream].append(downstream)
        all_nodes.add(upstream)
        all_nodes.add(downstream)
        downstream_nodes.add(downstream)
    return graph, all_nodes, downstream_nodes

def find_root_nodes(
    all_nodes: Set[str],
    downstream_nodes: Set[str],
    ignore_nodes: Optional[List[str]] = None,
    root_node: Optional[str] = None
) -> List[str]:
    """
    Identify the root nodes of a directed graph.

    A root node is defined as a node that does not appear as a downstream node in any edge,
    and is not in the ignore_nodes list. If a specific root_node is provided, only that node is returned.

    Parameters
    ----------
    all_nodes : Set[str]
        The set of all node names in the graph.
    downstream_nodes : Set[str]
        The set of all nodes that appear as downstream nodes in any edge.
    ignore_nodes : Optional[List[str]], optional
        A list of node names to ignore as possible roots. Defaults to None.
    root_node : Optional[str], optional
        If provided, this node is returned as the only root node.

    Returns
    -------
    List[str]
        A list of root node names.

    Examples
    --------
    >>> all_nodes = {'A', 'B', 'C'}
    >>> downstream_nodes = {'B', 'C'}
    >>> find_root_nodes(all_nodes, downstream_nodes)
    ['A']
    """
    if ignore_nodes is None:
        ignore_nodes = []
    if root_node is not None:
        return [root_node]
    return [node for node in all_nodes if node not in downstream_nodes and node not in ignore_nodes]

def find_all_paths(
    graph: Dict[str, List[str]],
    start_node: str,
    ignore_nodes: Optional[List[str]] = None
) -> List[List[str]]:
    """
    Find all possible acyclic paths starting from a given node in a directed graph.

    Parameters
    ----------
    graph : Dict[str, List[str]]
        The adjacency list representation of the graph.
    start_node : str
        The node from which to start searching for paths.
    ignore_nodes : Optional[List[str]], optional
        A list of node names to ignore during traversal. Defaults to None.

    Returns
    -------
    List[List[str]]
        A list of paths, where each path is a list of node names (strings) from the start_node to a destination node.
        Each path has at least two nodes (start and destination).

    Notes
    -----
    - Cycles are avoided: a node is not revisited in the same path.
    - Nodes in ignore_nodes are not included in any path.

    Examples
    --------
    >>> graph = {'A': ['B', 'C'], 'B': ['C'], 'C': []}
    >>> find_all_paths(graph, 'A')
    [['A', 'B'], ['A', 'B', 'C'], ['A', 'C']]
    """
    if ignore_nodes is None:
        ignore_nodes = []
    paths = []

    def dfs(current_node, current_path):
        if current_node in ignore_nodes:
            return
        new_path = current_path + [current_node]
        if len(new_path) > 1:
            paths.append(new_path)
        for neighbor in graph.get(current_node, []):
            if neighbor not in new_path and neighbor not in ignore_nodes:
                dfs(neighbor, new_path)
    dfs(start_node, [])
    return paths

def group_paths_by_destination(
    edges: list,
    ignore_nodes: list = ["core_metadata_collection"],
    root_node: Optional[str] = None
) -> Dict[str, List[PathInfo]]:
    """
    Find and group all possible acyclic paths in a directed graph by their destination node.

    For each destination node, all unique paths from any root node (or a specified root_node) to that destination
    are collected, ignoring any nodes in ignore_nodes.

    Parameters
    ----------
    edges : list of tuple
        List of (upstream, downstream) node pairs representing the directed edges of the graph.
    ignore_nodes : list, optional
        List of node names to ignore in the graph and in path traversal. Defaults to ["core_metadata_collection"].
    root_node : Optional[str], optional
        If provided, only paths starting from this node are considered as root paths.

    Returns
    -------
    Dict[str, List[PathInfo]]
        A dictionary mapping each destination node name to a list of PathInfo objects,
        each representing a unique path from a root node to that destination.

    Examples
    --------
    >>> edges = [('A', 'B'), ('B', 'C')]
    >>> group_paths_by_destination(edges)
    {'B': [PathInfo(path=['A', 'B'], steps=1)], 'C': [PathInfo(path=['A', 'B', 'C'], steps=2)]}
    """
    graph, all_nodes, downstream_nodes = build_graph(edges, ignore_nodes)
    root_nodes = find_root_nodes(all_nodes, downstream_nodes, ignore_nodes, root_node)
    print("Graph root node(s):", root_nodes)

    structured_results = defaultdict(list)
    for node in root_nodes:
        if node not in ignore_nodes:
            all_paths = find_all_paths(graph, node, ignore_nodes)
            for path in all_paths:
                destination_node = path[-1]
                path_info = PathInfo(path=path, steps=len(path) - 1)
                structured_results[destination_node].append(path_info)
    return dict(structured_results)

def get_min_node_path(
    edges: list,
    target_node: str,
    ignore_nodes: list = ["core_metadata_collection"],
    root_node: Optional[str] = None
) -> PathInfo:
    """
    Find the shortest path from a root node (or specified root_node) to a target node in a directed graph.

    Parameters
    ----------
    edges : list of tuple
        List of (upstream, downstream) node pairs representing the directed edges of the graph.
    target_node : str
        The destination node for which the shortest path is sought.
    ignore_nodes : list, optional
        List of node names to ignore in the graph and in path traversal. Defaults to ["core_metadata_collection"].
    root_node : Optional[str], optional
        If provided, only paths starting from this node are considered as root paths.

    Returns
    -------
    PathInfo
        The PathInfo object representing the shortest path from a root node to the target_node.

    Raises
    ------
    ValueError
        If no path exists from any root node to the target_node.

    Examples
    --------
    >>> edges = [('A', 'B'), ('B', 'C')]
    >>> get_min_node_path(edges, 'C')
    PathInfo(path=['A', 'B', 'C'], steps=2)
    """
    graph, all_nodes, downstream_nodes = build_graph(edges, ignore_nodes)
    root_nodes = find_root_nodes(all_nodes, downstream_nodes, ignore_nodes, root_node)
    all_paths_by_dest = group_paths_by_destination(edges, ignore_nodes=ignore_nodes, root_node=root_node)
    all_paths = all_paths_by_dest.get(target_node, [])
    root_paths = [p for p in all_paths if p.path and p.path[0] in root_nodes]
    if not root_paths:
        raise ValueError(f"No path from any root node to {target_node}")
    return min(root_paths, key=lambda path: path.steps)

In [ ]:
grouped_paths = group_paths_by_destination(dd.get_all_node_pairs())
grouped_paths

In [ ]:
get_min_node_path(dd.get_all_node_pairs(), 'variant_workflow', root_node="subject").path

In [ ]:
dd.get_all_node_pairs()

In [ ]:
edges = [
    ("project", "subject"),
    ("subject", "sample"),
]
print(f"Here are the edges")
print(edges)

def build_graph(
    edges,
    ignore_nodes=None
):
    """
    Build an adjacency list representation of a directed graph from a list of edges.

    Parameters
    ----------
    edges : List[Tuple[str, str]]
        A list of (upstream, downstream) node pairs representing directed edges in the graph.
    ignore_nodes : Optional[List[str]], optional
        A list of node names to ignore when building the graph. Edges involving these nodes are skipped.
        Defaults to None.

    Returns
    -------
    graph : Dict[str, List[str]]
        The adjacency list representation of the graph, mapping each node to a list of its downstream neighbors.
    all_nodes : Set[str]
        The set of all node names present in the graph (including both upstream and downstream nodes).
    downstream_nodes : Set[str]
        The set of all nodes that appear as downstream nodes in any edge.
    """
    from collections import defaultdict
    if ignore_nodes is None:
        ignore_nodes = []
    graph = defaultdict(list)
    all_nodes = set()
    downstream_nodes = set()
    for upstream, downstream in edges:
        if upstream in ignore_nodes or downstream in ignore_nodes:
            continue
        graph[upstream].append(downstream)
        all_nodes.add(upstream)
        all_nodes.add(downstream)
        downstream_nodes.add(downstream)
    return graph, all_nodes, downstream_nodes

# Example usage with a common edge example:


graph, all_nodes, downstream_nodes = build_graph(edges)
print("build_graph example:")
print("graph:", dict(graph))
print("all_nodes:", all_nodes)
print("downstream_nodes:", downstream_nodes)
print()


def find_root_node(
    all_nodes,
    downstream_nodes,
    ignore_nodes=None,
    root_node=None
):
    """
    Identify the root nodes of a directed graph.

    A root node is defined as a node that does not appear as a downstream node in any edge,
    and is not in the ignore_nodes list. If a specific root_node is provided, only that node is returned.

    Returns
    -------
    List[str]
        A list of root node names.
    """
    if ignore_nodes is None:
        ignore_nodes = []
    if root_node is not None:
        return [root_node]
    return [node for node in all_nodes if node not in downstream_nodes and node not in ignore_nodes]

# Example usage with a common edge example:
all_nodes = {"project", "subject", "sample"}
downstream_nodes = {"subject", "sample"}
roots = find_root_node(all_nodes, downstream_nodes)
print("find_root_node example:")
print("roots:", roots)
print()


def find_all_paths(
    graph,
    start_node,
    ignore_nodes=None
):
    """
    Find all possible acyclic paths starting from a given node in a directed graph.

    Returns
    -------
    List[List[str]]
        A list of paths, where each path is a list of node names (strings) from the start_node to a destination node.
        Each path has at least two nodes (start and destination).
    """
    if ignore_nodes is None:
        ignore_nodes = []
    paths = []

    def dfs(current_node, current_path):
        if current_node in ignore_nodes:
            return
        new_path = current_path + [current_node]
        if len(new_path) > 1:
            paths.append(new_path)
        for neighbor in graph.get(current_node, []):
            if neighbor not in new_path and neighbor not in ignore_nodes:
                dfs(neighbor, new_path)
    dfs(start_node, [])
    return paths

# Example usage with a common edge example:
graph = {'project': ['subject'], 'subject': ['sample'], 'sample': []}
all_paths = find_all_paths(graph, 'project')
print("find_all_paths example:")
print("all_paths:", all_paths)
print()


from collections import namedtuple, defaultdict

PathInfo = namedtuple("PathInfo", ["path", "steps"])

def group_paths_by_destination(
    edges,
    ignore_nodes=["core_metadata_collection"],
    root_node=None
):
    """
    Find and group all possible acyclic paths in a directed graph by their destination node.

    For each destination node, all unique paths from any root node (or a specified root_node) to that destination
    are collected, ignoring any nodes in ignore_nodes.

    Returns
    -------
    Dict[str, List[PathInfo]]
        A dictionary mapping each destination node name to a list of PathInfo objects,
        each representing a unique path from a root node to that destination.
    """
    graph, all_nodes, downstream_nodes = build_graph(edges, ignore_nodes)
    root_nodes = find_root_node(all_nodes, downstream_nodes, ignore_nodes, root_node)
    print("Graph root node(s):", root_nodes)

    structured_results = defaultdict(list)
    for node in root_nodes:
        if node not in ignore_nodes:
            all_paths = find_all_paths(graph, node, ignore_nodes)
            for path in all_paths:
                destination_node = path[-1]
                path_info = PathInfo(path=path, steps=len(path) - 1)
                structured_results[destination_node].append(path_info)
    return dict(structured_results)

# Example usage with a common edge example:
edges = [
    ("project", "subject"),
    ("subject", "sample"),
]
grouped = group_paths_by_destination(edges)
print("group_paths_by_destination example:")
for dest, paths in grouped.items():
    print(f"{dest}: {paths}")
print()


def get_min_node_path(
    edges,
    target_node,
    ignore_nodes=["core_metadata_collection"],
    root_node=None
):
    """
    Find the shortest path from a root node (or specified root_node) to a target node in a directed graph.

    Returns
    -------
    PathInfo
        The PathInfo object representing the shortest path from a root node to the target_node.

    Raises
    ------
    ValueError
        If no path exists from any root node to the target_node.
    """
    graph, all_nodes, downstream_nodes = build_graph(edges, ignore_nodes)
    root_nodes = find_root_node(all_nodes, downstream_nodes, ignore_nodes, root_node)
    all_paths_by_dest = group_paths_by_destination(edges, ignore_nodes=ignore_nodes, root_node=root_node)
    all_paths = all_paths_by_dest.get(target_node, [])
    root_paths = [p for p in all_paths if p.path and p.path[0] in root_nodes]
    if not root_paths:
        raise ValueError(f"No path from any root node to {target_node}")
    return min(root_paths, key=lambda path: path.steps)

# Example usage with a common edge example:
edges = [
    ("project", "subject"),
    ("subject", "sample"),
]
min_path = get_min_node_path(edges, 'sample')
print("get_min_node_path example:")
print("min_path:", min_path)
print()


In [1]:
from gen3_validator.resolve_schema import ResolveSchema


resolve_schema = ResolveSchema("../tests/schema/gen3_test_schema.json")
resolve_schema.resolve_schema()
resolve_schema.schema.keys()

dict_keys(['demographic.yaml', 'project.yaml', 'serum_marker_assay.yaml', 'alignment_workflow.yaml', 'imaging_file.yaml', 'lipidomics_assay.yaml', 'metabolomics_file.yaml', 'acknowledgement.yaml', 'medical_history.yaml', '_definitions.yaml', '_settings.yaml', 'blood_pressure_test.yaml', 'genomics_assay.yaml', 'variant_file.yaml', 'program.yaml', 'serum_marker_file.yaml', 'proteomics_assay.yaml', 'sample.yaml', 'unaligned_reads_file.yaml', '_terms.yaml', 'aligned_reads_index_file.yaml', 'variant_workflow.yaml', 'proteomics_file.yaml', 'exposure.yaml', 'metabolomics_assay.yaml', 'lipidomics_mapping_file.yaml', 'lipidomics_file.yaml', 'aligned_reads_file.yaml', 'lab_result.yaml', 'medication.yaml', 'publication.yaml', 'subject.yaml', 'core_metadata_collection.yaml'])

In [ ]:
resolve_schema.return_resolved_schema("_definitions.yaml")

In [ ]:
resolve_schema.schema_list_resolved


In [ ]:
resolve_schema.schema_list_resolved

In [ ]:
resolve_schema.schema_list

In [ ]:
from collections import defaultdict

def find_all_paths_and_steps(self, edges: list) -> dict:
    """
    Finds all possible paths in a directed graph and calculates the number of steps for each.

    This method builds a graph from the list of edges and performs a Depth-First Search (DFS)
    from each source node (a node with no incoming edges). It records every intermediate
    path and its corresponding length.

    For example, for a graph A -> B -> C, this function will identify and record:
    - ('A', 'B'): 1
    - ('A', 'B', 'C'): 2

    :param edges: A list of tuples, where each tuple is a node pair (upstream, downstream).
    :type edges: list

    :return: A dictionary where keys are tuples representing the paths and values are the
             number of steps (edges) in that path.
    :rtype: dict
    """
    # Step 1: Build the graph and identify source nodes, similar to your method.
    graph = defaultdict(list)
    in_degree = defaultdict(int)
    all_nodes = set()

    for upstream, downstream in edges:
        graph[upstream].append(downstream)
        in_degree[downstream] += 1
        all_nodes.add(upstream)
        all_nodes.add(downstream)
    
    # Identify source nodes (nodes with an in-degree of 0).
    source_nodes = [node for node in all_nodes if in_degree[node] == 0]

    # This dictionary will store the final results.
    all_paths = {}

    # Step 2: Define the recursive DFS traversal function.
    def dfs(current_path):
        """Recursively explores the graph and records paths."""
        node = current_path[-1]

        # Record every path segment that has at least one step.
        if len(current_path) > 1:
            all_paths[tuple(current_path)] = len(current_path) - 1
        
        # If we reach a node with no outgoing edges, the path ends here.
        if node not in graph:
            return
        
        # Continue the traversal for all neighbors.
        for neighbor in graph[node]:
            # To prevent infinite loops in graphs with cycles, we don't revisit nodes
            # already in the current traversal path.
            if neighbor not in current_path:
                dfs(current_path + [neighbor])

    # Step 3: Start the DFS from each identified source node.
    for start_node in source_nodes:
        dfs([start_node])
        
    return all_paths



In [ ]:
from collections import defaultdict

def find_all_paths_and_steps(self, edges: list) -> dict:
    """
    Finds all possible paths in a directed graph and calculates the number of steps for each.

    This method builds a graph from the list of edges and performs a Depth-First Search (DFS)
    from each source node (a node with no incoming edges). It records every intermediate
    path and its corresponding length.

    For example, for a graph A -> B -> C, this function will identify and record:
    - ('A', 'B'): 1
    - ('A', 'B', 'C'): 2

    :param edges: A list of tuples, where each tuple is a node pair (upstream, downstream).
    :type edges: list

    :return: A dictionary where keys are tuples representing the paths and values are the
             number of steps (edges) in that path.
    :rtype: dict
    """
    # Step 1: Build the graph and identify source nodes, similar to your method.
    graph = defaultdict(list)
    in_degree = defaultdict(int)
    all_nodes = set()

    for upstream, downstream in edges:
        graph[upstream].append(downstream)
        in_degree[downstream] += 1
        all_nodes.add(upstream)
        all_nodes.add(downstream)
    
    # Identify source nodes (nodes with an in-degree of 0).
    source_nodes = [node for node in all_nodes if in_degree[node] == 0]

    # This dictionary will store the final results.
    all_paths = {}

    # Step 2: Define the recursive DFS traversal function.
    def dfs(current_path):
        """Recursively explores the graph and records paths."""
        node = current_path[-1]

        # Record every path segment that has at least one step.
        if len(current_path) > 1:
            all_paths[tuple(current_path)] = len(current_path) - 1
        
        # If we reach a node with no outgoing edges, the path ends here.
        if node not in graph:
            return
        
        # Continue the traversal for all neighbors.
        for neighbor in graph[node]:
            # To prevent infinite loops in graphs with cycles, we don't revisit nodes
            # already in the current traversal path.
            if neighbor not in current_path:
                dfs(current_path + [neighbor])

    # Step 3: Start the DFS from each identified source node.
    for start_node in source_nodes:
        dfs([start_node])
        
    return all_paths



In [ ]:
from collections import defaultdict

def find_all_paths_and_steps(self, edges: list) -> dict:
    """
    Finds all possible paths in a directed graph and calculates the number of steps for each.

    This method builds a graph from the list of edges and performs a Depth-First Search (DFS)
    from each source node (a node with no incoming edges). It records every intermediate
    path and its corresponding length.

    For example, for a graph A -> B -> C, this function will identify and record:
    - ('A', 'B'): 1
    - ('A', 'B', 'C'): 2

    :param edges: A list of tuples, where each tuple is a node pair (upstream, downstream).
    :type edges: list

    :return: A dictionary where keys are tuples representing the paths and values are the
             number of steps (edges) in that path.
    :rtype: dict
    """
    # Step 1: Build the graph and identify source nodes, similar to your method.
    graph = defaultdict(list)
    in_degree = defaultdict(int)
    all_nodes = set()

    for upstream, downstream in edges:
        graph[upstream].append(downstream)
        in_degree[downstream] += 1
        all_nodes.add(upstream)
        all_nodes.add(downstream)
    
    # Identify source nodes (nodes with an in-degree of 0).
    source_nodes = [node for node in all_nodes if in_degree[node] == 0]

    # This dictionary will store the final results.
    all_paths = {}

    # Step 2: Define the recursive DFS traversal function.
    def dfs(current_path):
        """Recursively explores the graph and records paths."""
        node = current_path[-1]

        # Record every path segment that has at least one step.
        if len(current_path) > 1:
            all_paths[tuple(current_path)] = len(current_path) - 1
        
        # If we reach a node with no outgoing edges, the path ends here.
        if node not in graph:
            return
        
        # Continue the traversal for all neighbors.
        for neighbor in graph[node]:
            # To prevent infinite loops in graphs with cycles, we don't revisit nodes
            # already in the current traversal path.
            if neighbor not in current_path:
                dfs(current_path + [neighbor])

    # Step 3: Start the DFS from each identified source node.
    for start_node in source_nodes:
        dfs([start_node])
        
    return all_paths



In [ ]:
from gen3_validator.dict import DataDictionary
dd = DataDictionary(schema_path = "../tests/schema/gen3_test_schema.json")
dd.parse_schema()
dd.get_nodes()
